# U5.1 — Preparación de PySpark y exploración de NYC Taxi

**Materia:** Tópicos de Big Data  
**Objetivo:** comprobar PySpark en VS Code y explorar una muestra real de NYC Yellow Taxi sin guardar el dataset completo en la computadora.

**Alcance:** instalación, SparkSession, DataFrames y lectura remota de Parquet. Sin Databricks, limpieza ni modelos de ML.

**Fuente:** [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), Yellow Taxi, enero de 2025.

**Nota:** Spark se ejecuta localmente. PyArrow leerá por HTTP los fragmentos del Parquet necesarios; esto **no equivale a procesamiento distribuido remoto** y sí consume red y memoria. No se crea un archivo de datos local.


## 1. Preparar y comprobar el entorno

Selecciona el kernel Python 3.12 de `.venv` en VS Code. Si faltan paquetes, instala en la terminal:

```bash
python -m pip install pyspark ipykernel pyarrow
```

Comprueba Python, PySpark, PyArrow y Java ejecutando la siguiente celda. **No es necesario instalar `fsspec` ni `aiohttp`.**


In [0]:
import sys
import subprocess
import pyspark
import pyarrow

print("Python:", sys.version.split()[0])
print("PySpark:", pyspark.__version__)
print("PyArrow:", pyarrow.__version__)
java = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java:", (java.stderr or java.stdout).splitlines()[0])


## 2. Crear una SparkSession

`local[*]` utiliza los núcleos disponibles de esta computadora.


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

print("Versión de Spark:", spark.version)
print("SparkSession creada correctamente")

## 3. Probar un DataFrame pequeño

Comprobamos que Spark puede procesar registros y mostrar su esquema.


In [0]:
datos = [
    (1, 3.5, 15.0),
    (2, 8.2, 32.5),
    (3, 1.8, 10.0)
]

columnas = ["viaje_id", "distancia", "tarifa"]

df = spark.createDataFrame(datos, columnas)

df.show()
df.printSchema()

## 4. Conocer la fuente remota

El archivo original está publicado en **Parquet**, un formato columnar. Consultaremos su estructura mediante **PyArrow**, leyendo rangos HTTP del archivo sin guardarlo en disco.

**¿Por qué no `spark.read.parquet(URL)`?** Spark local no interpreta directamente una URL HTTPS común como un sistema de archivos Parquet. Para esta práctica introductoria, PyArrow leerá una parte del archivo y luego Spark procesará una muestra pequeña.

El servidor debe admitir solicitudes HTTP `Range`. Si no las admite o falla la conexión, aparecerá un error claro.


In [0]:
import os
import pyarrow.parquet as pq

if os.environ.get("DATABRICKS_RUNTIME_VERSION"):
    # Archivo subido a Databricks
    RUTA_NYC = "/Volumes/workspace/default/nyc_taxo/yellow_tripdata_2025-01.parquet"
    parquet = pq.ParquetFile(RUTA_NYC)
else:
    # Lectura HTTP por rangos en su computadora
    archivo_remoto = io.BufferedReader(
        ArchivoHTTPPorRangos(URL_NYC),
        buffer_size=1024 * 1024
    )
    parquet = pq.ParquetFile(archivo_remoto)

print("Grupos de filas:", parquet.num_row_groups)
print("Filas totales declaradas:", parquet.metadata.num_rows)
print("Columnas:", parquet.schema.names)

## 5. Leer una muestra sin descargar el dataset completo

Leeremos **solo tres columnas del primer grupo de filas** y tomaremos sus primeros 1,000 registros. Un grupo de filas puede contener muchos más registros: PyArrow debe transferir las partes necesarias de esas columnas, por lo que **la transferencia puede ser mayor que 1,000 filas**.

El resultado sí pasa a la memoria local para crear el DataFrame de PySpark.


In [0]:
columnas_nyc = ["VendorID", "trip_distance", "total_amount"]

# PyArrow lee las columnas seleccionadas del primer grupo de filas.
tabla_arrow = parquet.read_row_group(0, columns=columnas_nyc)
muestra = tabla_arrow.slice(0, 1000)

# Convertimos la muestra a registros Python para crear un DataFrame Spark.
df_taxi = spark.createDataFrame(muestra.to_pylist())

print("Registros de la muestra:", df_taxi.count())
df_taxi.printSchema()
df_taxi.show(5)


## 6. Primera exploración con PySpark

Utilizamos operaciones básicas: selección, filtro y agregación. **Los resultados corresponden a la muestra, no a todos los viajes de enero.**


In [0]:
from pyspark.sql import functions as F

# Mostrar viajes con distancia positiva.
df_taxi.filter(F.col("trip_distance") > 0).show(5)

# Resumen sencillo de la muestra.
df_taxi.select(
    F.count("*").alias("viajes_muestra"),
    F.round(F.avg("trip_distance"), 2).alias("distancia_promedio"),
    F.round(F.avg("total_amount"), 2).alias("importe_promedio")
).show()


## 7. Comprobación y reflexión

1. ¿Qué hace `SparkSession`?
2. ¿Cuál es la diferencia entre `df.show()` y `df.printSchema()`?
3. ¿Qué biblioteca accedió al archivo Parquet remoto y cuál procesó la muestra?
4. ¿Por qué no podemos afirmar que procesamos todo el dataset de NYC Taxi?
5. ¿En qué computadora se ejecutaron las operaciones de Spark?

**Resultado esperado:** una sesión de Spark funcional, metadatos del Parquet remoto y un DataFrame de muestra procesado localmente, sin guardar el dataset en disco.
